# Retake coach: freeze the detector, test it once on unseen photos, add the energy guard

**Cost: $0** in API credits. The last cell (Muse Spark wording) is **off** unless you set `RUN_MUSE = True`.
If you switch it on, it stops at 8 requests or $0.08, whichever comes first.
**Meta models:** DINOv2 (detector and guard, runs on the Colab GPU) and, optionally, Muse Spark (wording only).
**Runtime:** GPU (T4 or L4). About 10 minutes, mostly downloading 450 DeepDRiD photos.

What it answers, each on photos the heads never trained on:
1. **Held-out test.** Heads are trained on DeepDRiD *training* photos and thresholds picked on *validation*.
   They are scored once on the 400 DeepDRiD *test* photos (Challenge 2 labels), which no earlier notebook used.
2. **Other hospitals.** Trained on one DeepDRiD site (Nicheng), scored on the other two.
3. **New camera.** IDRiD photos come from another country and camera and were picked for grading. How often does the tool wrongly
   ask for a retake? On IDRiD photos damaged on purpose, does it name the right problem?
4. **Guard (energy).** It uses a Gaussian energy over ordinary fundus photos. Does it accept fundus photos and turn away
   widefield images and everyday pictures?
5. It saves the tool's model to Drive as `quality/coach_model.npz` for the demo app.

**Send me:** `quality/build_report.txt` and `quality/coach_examples.png` (both in your Drive folder).

In [ ]:
import os, glob, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'
assert os.path.exists(f'{D}/quality/features.npz'), 'run quality_energy_test.ipynb first: it caches the DeepDRiD training and validation features'
DD = '/content/deepdrid'

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

# DeepDRiD: only the 400 test photos, the 50 widefield validation photos and the label files (no training images)
if not os.path.exists(f'{DD}/regular_fundus_images/Online-Challenge1&2-Evaluation/Images'):
    sh(f'rm -rf {DD} && git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/deepdrdoc/DeepDRiD.git {DD}')
    with open(f'{DD}/.git/info/sparse-checkout', 'w') as f:
        f.write('/regular_fundus_images/*/*.csv\n/regular_fundus_images/Online-Challenge1&2-Evaluation/\n'
                '/ultra-widefield_images/ultra-widefield-validation/\n')
    sh(f'cd {DD} && git config core.sparseCheckout true && git checkout -q')
if not glob.glob('/content/idrid/**/*.jpg', recursive=True):
    sh(f'unzip -q -o "{D}/A. Segmentation.zip" -d /content/idrid && unzip -q -o "{D}/B. Disease Grading.zip" -d /content/idrid')
print(len(glob.glob(f'{DD}/regular_fundus_images/Online-Challenge1&2-Evaluation/Images/*/*.jpg')), 'DeepDRiD test photos |',
      len(glob.glob(f'{DD}/ultra-widefield_images/ultra-widefield-validation/Images/*/*.jpg')), 'widefield photos |',
      len(glob.glob('/content/idrid/**/*Original Images/*/*.jpg', recursive=True)), 'IDRiD photos')

The tool's code, in one file. The demo app imports the same file.

In [ ]:
%%writefile /content/coach_core.py
"""Retake coach: is this fundus photo good enough to grade, and if not, what should the photographer fix?

It judges photo quality only and makes no diagnosis.

- Detection: Meta's DINOv2 (facebook/dinov2-base, frozen) runs locally. Four small logistic heads read its
  features plus three simple image checks: not good enough, focus, aim, obstruction. The heads are trained on
  DeepDRiD's photo-quality labels (CC BY-SA 4.0, https://github.com/deepdrdoc/DeepDRiD).
- Guard: an energy on the same features (a Gaussian model of ordinary fundus photos, i.e. Mahalanobis distance)
  flags inputs that do not look like a regular fundus photo, such as a widefield image or a random picture.
- Wording: Muse Spark (Meta Model API) writes the instruction for the problems the heads found. A hard cap on
  requests and dollars applies; past the cap, or on any error, fixed templates are used.
"""
import base64
import io
import json

import numpy as np
from PIL import Image
from scipy import ndimage

S = 448
PROBLEMS = ("focus", "aim", "obstruction")
DESCRIBE = {
    "focus": "out of focus or hazy: the small vessels are not sharp",
    "aim": "badly aimed: the optic disc and the macula are not both near the centre of the field",
    "obstruction": "partly covered: eyelid or eyelash shadow, glare, reflections or dust over the retina",
}
TEMPLATES = {
    "focus": "Refocus until the small vessels look sharp, then retake.",
    "aim": "Re-aim so the optic disc and the macula are both near the centre, then retake.",
    "obstruction": "Ask the patient to open wide (lift the eyelid if needed), wipe the lens, then retake.",
}
GENERIC = "Not good enough to grade: check focus, aim and eyelids, then retake."
NOT_FUNDUS = "This does not look like a regular fundus photo. Check the camera mode or the file."
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])


# ---------- image preparation (identical to the pipeline the heads were trained on) ----------

def _open(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if isinstance(src, np.ndarray):
        return Image.fromarray(src).convert("RGB")
    return Image.open(src).convert("RGB")


def load_square(src):
    """Crop to the retina, pad to a square, resize to 448 px. Returns float32 in [0, 1]. src: path, PIL image or uint8 array."""
    img = _open(src)
    img.thumbnail((1024, 1024))
    a = np.asarray(img)
    ys, xs = np.nonzero(a.max(-1) > 15)
    if len(ys):                                   # an all-black frame keeps its full size
        a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = a.shape[:2]
    side = max(h, w)
    sq = np.zeros((side, side, 3), np.uint8)
    sq[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = a
    return np.asarray(Image.fromarray(sq).resize((S, S), Image.BILINEAR)).astype(np.float32) / 255


def retina_mask(x):
    return x.max(-1) > 15 / 255


def simple_checks(x):
    """Sharpness (Laplacian variance), brightness and contrast of the green channel inside the retina."""
    g, m = x[..., 1], ndimage.binary_erosion(retina_mask(x), iterations=8)
    if m.sum() < 100:                             # nearly black frame: measure the whole picture
        m = np.ones_like(m)
    return [ndimage.laplace(g)[m].var(), g[m].mean(), g[m].std()]


class Encoder:
    """Frozen DINOv2: class token + mean patch token (1,536 numbers per photo)."""

    def __init__(self, name="facebook/dinov2-base", device=None):
        import torch
        from transformers import AutoModel
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModel.from_pretrained(name).to(self.device, self.dtype).eval()

    def __call__(self, xs, bs=32):
        out = []
        with self.torch.no_grad():
            for i in range(0, len(xs), bs):
                b = self.torch.tensor((np.stack(xs[i:i + bs]) - MEAN) / STD).permute(0, 3, 1, 2)
                h = self.model(pixel_values=b.to(self.device, self.dtype)).last_hidden_state
                out.append(self.torch.cat([h[:, 0], h[:, 1:].mean(1)], 1).float().cpu())
        return self.torch.cat(out).numpy()


def describe(srcs, encoder, bs=32, workers=8):
    """Photos -> (DINOv2 features, simple checks). Loads in parallel, encodes in batches."""
    from concurrent.futures import ThreadPoolExecutor
    X, C = [], []
    for i in range(0, len(srcs), bs):
        with ThreadPoolExecutor(workers) as ex:
            xs = list(ex.map(load_square, srcs[i:i + bs]))
        X.append(encoder(xs, bs))
        C += [simple_checks(x) for x in xs]
    return np.concatenate(X), np.array(C)


# ---------- detector heads and energy guard ----------

def _sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class Coach:
    HEADS = ("retake",) + PROBLEMS
    RULES = ("overall", "problems", "both")   # what triggers a retake: the overall head, any problem head, or both

    def __init__(self, mu, sd, W, b, thr, g_mean=None, g_white=None, g_thr=None, rule="overall"):
        assert rule in self.RULES
        self.mu, self.sd, self.W, self.b, self.thr, self.rule = mu, sd, W, b, np.asarray(thr, float), rule
        self.g_mean, self.g_white, self.g_thr = g_mean, g_white, g_thr

    @classmethod
    def fit(cls, X, checks, labels, C=0.05):
        """labels: dict head -> bool array. Thresholds start at 0.5; set coach.thr from validation photos."""
        from sklearn.linear_model import LogisticRegression
        mu, sd = checks.mean(0), checks.std(0) + 1e-12
        F = np.c_[X, (checks - mu) / sd]
        W, b = [], []
        for h in cls.HEADS:
            clf = LogisticRegression(C=C, max_iter=3000).fit(F, labels[h])
            W.append(clf.coef_[0])
            b.append(clf.intercept_[0])
        return cls(mu, sd, np.array(W), np.array(b), np.full(len(cls.HEADS), 0.5))

    def probs(self, X, checks):
        """(n, 4) probabilities in HEADS order."""
        return _sigmoid(np.c_[X, (checks - self.mu) / self.sd] @ self.W.T + self.b)

    def set_guard(self, X_ref, X_cal, quantile=0.99):
        """Energy guard fitted on ordinary fundus photos (X_ref); threshold = the 99th percentile on other ones (X_cal)."""
        from sklearn.covariance import LedoitWolf
        lw = LedoitWolf().fit(np.asarray(X_ref, np.float64))
        lam, V = np.linalg.eigh(lw.covariance_)               # whiten from the covariance itself:
        lam = np.maximum(lam, lam.max() * 1e-6)               # tiny directions are floored, never inverted to inf
        self.g_mean, self.g_white = lw.location_, V / np.sqrt(lam)
        self.g_thr = float(np.quantile(self.energy(X_cal), quantile))
        return self

    def energy(self, X):
        """Mahalanobis energy per feature dimension: about 1 for typical fundus photos, large for unfamiliar inputs."""
        return (((X - self.g_mean) @ self.g_white) ** 2).sum(1) / X.shape[1]

    def verdict(self, X, checks, max_problems=2):
        P = self.probs(X, checks)
        E = self.energy(X) if self.g_mean is not None else np.zeros(len(X))
        out = []
        for p, e in zip(P, E):
            fundus = self.g_thr is None or bool(e <= self.g_thr)
            margin = _logit(p[1:]) - _logit(self.thr[1:])          # how far each problem head is past its threshold
            overall, any_problem = bool(p[0] >= self.thr[0]), bool((margin > 0).any())
            retake = {"overall": overall, "problems": any_problem, "both": overall and any_problem}[self.rule]
            order = [int(i) for i in np.argsort(-margin)]
            problems = [PROBLEMS[i] for i in order if margin[i] > 0][:max_problems] if retake else []
            if retake and not problems:                              # always give one concrete thing to fix
                problems = [PROBLEMS[order[0]]]
            out.append(dict(looks_like_fundus=fundus, retake=retake and fundus, problems=problems if fundus else [],
                            p_not_good=float(p[0]), scores={k: float(v) for k, v in zip(PROBLEMS, p[1:])},
                            energy=float(e)))
        return out

    def save(self, path):
        g = {} if self.g_mean is None else dict(g_mean=self.g_mean, g_white=self.g_white.astype(np.float32), g_thr=self.g_thr)
        np.savez(path, mu=self.mu, sd=self.sd, W=self.W, b=self.b, thr=self.thr, heads=np.array(self.HEADS),
                 rule=np.array(self.rule), **g)

    @classmethod
    def load(cls, path):
        z = np.load(path)
        assert tuple(z["heads"]) == cls.HEADS
        g = dict(g_mean=z["g_mean"], g_white=z["g_white"].astype(np.float64), g_thr=float(z["g_thr"])) if "g_mean" in z else {}
        return cls(z["mu"], z["sd"], z["W"], z["b"], z["thr"], rule=str(z["rule"]) if "rule" in z else "overall", **g)


# ---------- wording: Muse Spark, capped, with templates as the fallback ----------

PROMPT = """You coach a technician who takes retinal (fundus) photos for eye screening. An automatic photo-quality
check found that this photo needs a retake because it is:
{found}
Write ONE instruction for the technician, at most 25 words, that fixes only these problems. Look at the photo only
to say where (for example which side a shadow or glare is on). Do not mention disease or diagnosis.
Reply with JSON only: {{"instruction": "..."}}"""


def template(problems):
    return " ".join(TEMPLATES[p] for p in problems) if problems else GENERIC


def _data_url(src, px):
    img = _open(src)
    img.thumbnail((px, px))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def _json_field(text, key):
    try:
        return str(json.loads(text[text.index("{"):text.rindex("}") + 1])[key]).strip() or None
    except (ValueError, KeyError, TypeError):
        return None


class Wording:
    PRICE_IN, PRICE_OUT = 1.25e-6, 4.25e-6       # muse-spark-1.3, Standard mode, $ per token

    def __init__(self, client=None, model="muse-spark-1.3", effort="minimal", max_requests=20, max_usd=0.10, px=768):
        self.client, self.model, self.effort, self.px = client, model, effort, px
        self.max_requests, self.max_usd = max_requests, max_usd
        self.n, self.spent, self.worst = 0, 0.0, 0.01  # assume a call may cost 1 cent until a dearer one is seen

    def can_call(self):
        return self.client is not None and self.n < self.max_requests and self.spent + self.worst <= self.max_usd

    def __call__(self, src, problems):
        if not problems or not self.can_call():
            return dict(text=template(problems), source="template", cost=0.0)
        found = "\n".join(f"- {DESCRIBE[p]}" for p in problems)
        self.n += 1
        try:
            resp = self.client.chat.completions.create(
                model=self.model, extra_body={"reasoning_effort": self.effort},
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": PROMPT.format(found=found)},
                    {"type": "image_url", "image_url": {"url": _data_url(src, self.px)}}]}])
        except Exception as e:  # network, quota, bad key: the technician still gets an instruction
            return dict(text=template(problems), source=f"template (Muse Spark error: {type(e).__name__})", cost=0.0)
        cost = resp.usage.prompt_tokens * self.PRICE_IN + resp.usage.completion_tokens * self.PRICE_OUT
        self.spent += cost
        self.worst = max(self.worst, cost)
        text = _json_field(resp.choices[0].message.content or "", "instruction")
        return dict(text=text or template(problems), source="muse-spark" if text else "template (unparsed reply)", cost=cost)


In [ ]:
import importlib, ast
import numpy as np, pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
sys.path.insert(0, '/content')
import coach_core as cc
importlib.reload(cc)

os.makedirs(f'{D}/quality', exist_ok=True)
REPORT = f'{D}/quality/build_report.txt'
open(REPORT, 'w').close()
def say(*a):
    s = ' '.join(str(x) for x in a)
    print(s)
    with open(REPORT, 'a') as f:
        f.write(s + '\n')

rng = np.random.default_rng(0)
S = cc.S
def blur(x, r):
    from PIL import ImageFilter
    return np.asarray(Image.fromarray((x * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(r))).astype(np.float32) / 255

def corrupt(x, kind):   # same damage as quality_energy_test.ipynb
    m = cc.retina_mask(x)[..., None]; yy, xx = np.mgrid[:S, :S] / S
    if kind == 'blur':      y = blur(x, rng.uniform(2, 5))
    elif kind == 'haze':    y = blur(x.mean((0, 1)) + (x - x.mean((0, 1))) * rng.uniform(0.35, 0.6), 1)
    elif kind == 'dark':    y = x * rng.uniform(0.3, 0.55)
    elif kind == 'washout': a = rng.uniform(0.35, 0.6); y = x * (1 - a) + 0.9 * a
    elif kind == 'shadow':  t = rng.uniform(0.2, 0.45); y = x * np.clip((yy - t) / 0.08, 0.15, 1)[..., None]
    elif kind == 'glare':
        y = x.copy()
        for _ in range(rng.integers(1, 4)):
            ang, r = rng.uniform(0, 2 * np.pi), rng.uniform(0.05, 0.15)
            cy, cx = 0.5 + 0.38 * np.sin(ang), 0.5 + 0.38 * np.cos(ang)
            y = y + rng.uniform(0.5, 0.9) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2))[..., None]
    elif kind == 'aim':
        c = int(S * rng.uniform(0.6, 0.7)); ox, oy = rng.choice([0, S - c]), rng.integers(0, S - c)
        y = np.asarray(Image.fromarray((x[oy:oy + c, ox:ox + c] * 255).astype(np.uint8)).resize((S, S))).astype(np.float32) / 255
    return (np.clip(y, 0, 1) * m).astype(np.float32)
GROUPS = {'focus': ['blur', 'haze'], 'exposure': ['dark', 'washout'], 'obstruction': ['shadow', 'glare'], 'aim': ['aim']}

# cached DeepDRiD training + validation features (from quality_energy_test.ipynb)
F = np.load(f'{D}/quality/features.npz', allow_pickle=True)
dd = pd.DataFrame(F['dd_labels']); X, chk = F['dd'], F['dd_checks']
T = f'{DD}/regular_fundus_images/Online-Challenge1&2-Evaluation'
te = pd.read_excel(f'{T}/Challenge2_labels.xlsx')
te_paths = [f"{T}/Images/{i.split('_')[0]}/{i}.jpg" for i in te.image_id]
uwf_paths = sorted(glob.glob(f'{DD}/ultra-widefield_images/ultra-widefield-validation/Images/*/*.jpg'))
idr_paths = sorted(set(glob.glob('/content/idrid/**/*Original Images/*/*.jpg', recursive=True)))

New features: DeepDRiD test, widefield, IDRiD (as is and damaged), everyday pictures. Cached in Drive after the first run.

In [ ]:
cache = f'{D}/quality/build_features.npz'
if os.path.exists(cache):
    G = dict(np.load(cache, allow_pickle=True))
else:
    enc = cc.Encoder()
    G = {}
    G['te_X'], G['te_C'] = cc.describe(te_paths, enc)
    G['uwf_X'], G['uwf_C'] = cc.describe(uwf_paths, enc)
    G['idr_X'], G['idr_C'] = cc.describe(idr_paths, enc)
    # damage 150 IDRiD photos (from the part the guard never sees), one kind from each group
    base = [cc.load_square(p) for p in idr_paths[300:450]]
    cX, cC, groups = [], [], []
    for g, ks in GROUPS.items():   # one group at a time keeps memory small
        xs = [corrupt(x, rng.choice(ks)) for x in base]
        cX.append(enc(xs)); cC += [cc.simple_checks(x) for x in xs]; groups += [g] * len(xs)
    G['cor_X'], G['cor_C'], G['cor_group'] = np.concatenate(cX), np.array(cC), np.array(groups)
    # everyday pictures (scikit-image samples), a black frame, and one fundus photo from the web (skimage 'retina')
    import skimage.data as sd
    names, pics = [], []
    for n in ['astronaut', 'coffee', 'chelsea', 'rocket', 'hubble_deep_field', 'immunohistochemistry', 'camera', 'moon',
              'page', 'text', 'coins', 'horse', 'retina']:
        try:
            a = np.asarray(getattr(sd, n)())
        except Exception as e:
            print('skipped', n, type(e).__name__); continue
        if a.dtype == bool: a = a.astype(np.uint8) * 255
        if a.ndim == 2: a = np.stack([a] * 3, -1)
        names.append(n); pics.append(a[..., :3].astype(np.uint8))
    names.append('black frame'); pics.append(np.zeros((900, 1200, 3), np.uint8))
    G['ev_X'], G['ev_C'] = cc.describe(pics, enc); G['ev_names'] = np.array(names)
    np.savez(cache, **G)
print({k: v.shape for k, v in G.items() if k.endswith('_X')})

## 1. Held-out test
Heads are trained on DeepDRiD training photos. Thresholds come from validation only:
- **Retake:** catch at least 90% of the not-good-enough validation photos. A missed bad photo means the patient is called back.
  A needless retake costs about 30 seconds.
- **Each problem:** the threshold that best separates it on validation (Youden).

The test photos are scored once.

In [ ]:
def targets(df):
    return {'retake': (df['Overall quality'] == 0).values, 'focus': (df['Clarity'] <= 6).values,
            'aim': (df['Field definition'] <= 6).values, 'obstruction': (df['Artifact'] >= 6).values}

def auc_ci(y, s, n=1000):
    if y.min() == y.max(): return np.nan, np.nan, np.nan
    r = np.random.default_rng(1); boot = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): boot.append(roc_auc_score(y[i], s[i]))
    return roc_auc_score(y, s), *np.percentile(boot, [2.5, 97.5])

tr, va = (dd.split == 'train').values, (dd.split == 'val').values
Y, Yte = targets(dd), targets(te)
coach = cc.Coach.fit(X[tr], chk[tr], {k: v[tr] for k, v in Y.items()})
Pva, Pte = coach.probs(X[va], chk[va]), coach.probs(G['te_X'], G['te_C'])
thr = []
for j, h in enumerate(cc.Coach.HEADS):
    y, p = Y[h][va], Pva[:, j]
    if h == 'retake':
        thr.append(np.sort(p[y])[int(np.floor(0.10 * y.sum()))])
    else:
        fpr, tpr, th = roc_curve(y, p); thr.append(th[np.argmax(tpr - fpr)])
coach.thr = np.array(thr)

rows = []
for j, h in enumerate(cc.Coach.HEADS):
    y, p, t = Yte[h], Pte[:, j], coach.thr[j]
    auc, lo, hi = auc_ci(y, p)
    rows.append({'check': h, 'test photos with it': f'{y.sum()}/{len(y)}', 'AUROC test': round(auc, 3), '95% CI': f'{lo:.2f}-{hi:.2f}',
                 'AUROC validation': round(roc_auc_score(Y[h][va], Pva[:, j]), 3),
                 'sharpness only (test)': round(roc_auc_score(y, -G['te_C'][:, 0]), 3),
                 'sensitivity': round((p[y] >= t).mean(), 2), 'specificity': round((p[~y] < t).mean(), 2)})
say('== 1. Held-out DeepDRiD test (400 photos, never used before) ==')
say(pd.DataFrame(rows).to_string(index=False))

V = coach.verdict(G['te_X'], G['te_C'])
flag, bad = np.array([v['retake'] for v in V]), Yte['retake']
truth = [{k for k in cc.PROBLEMS if Yte[k][i]} for i in range(len(te))]
say(f"\nRetake asked for {flag.mean():.0%} of test photos: caught {(flag & bad).sum()}/{bad.sum()} not-good-enough photos, "
    f"sent back {(flag & ~bad).sum()}/{(~bad).sum()} good ones.")
caught = [i for i in np.where(flag & bad)[0] if truth[i]]
say(f"Caught photos with a labelled problem: the top named problem is a real one in {np.mean([V[i]['problems'][0] in truth[i] for i in caught]):.0%}, "
    f"at least one named problem is real in {np.mean([bool(set(V[i]['problems']) & truth[i]) for i in caught]):.0%} (n={len(caught)}); "
    f"{np.mean([len(V[i]['problems']) for i in np.where(flag)[0]]):.1f} problems named per retake on average.")
for k in cc.PROBLEMS:
    idx = [i for i in np.where(flag)[0] if truth[i] == {k}]
    if idx: say(f"  only {k:11s}: top named problem right {np.mean([V[i]['problems'][0] == k for i in idx]):.0%} (n={len(idx)})")

Same 20 photos Muse Spark judged in the pilot, side by side. These are validation photos, which picked the thresholds,
so read this as an illustration, not a test.

In [ ]:
mp = f'{D}/quality/muse_coach_pilot.csv'
if os.path.exists(mp):
    m = pd.read_csv(mp)
    def as_list(s):
        try: return list(ast.literal_eval(s))
        except Exception: return []
    m['problems'] = m.problems.apply(as_list)
    where = {i: n for n, i in enumerate(dd.image_id)}
    rows_ = [where[i] for i in m.image_id]
    Vm = coach.verdict(X[rows_], chk[rows_])
    out = []
    for g, part in m.groupby('truth', sort=False):
        ids = part.index
        heads_named = [Vm[i]['problems'] for i in ids]
        if g == 'good':
            out.append({'photos': f'good (n={len(ids)})', 'kept: heads': str(sum(not Vm[i]['retake'] for i in ids)),
                        'kept: Muse': str(int((part.retake == False).sum()))})
        else:
            out.append({'photos': f'{g} only (n={len(ids)})', 'retake: heads': str(sum(Vm[i]['retake'] for i in ids)),
                        'retake: Muse': str(int((part.retake == True).sum())),
                        'right problem first: heads': str(sum(bool(p) and p[0] == g for p in heads_named)),
                        'right problem named: Muse': str(sum(g in p for p in part.problems)),
                        'problems named, avg: heads': round(np.mean([len(p) for p in heads_named]), 1),
                        'problems named, avg: Muse': round(part.problems.apply(len).mean(), 1)})
    say('\n== Heads vs Muse Spark on the pilot\'s 20 validation photos ==')
    say(pd.DataFrame(out).fillna('').to_string(index=False))

## 2. Other hospitals: trained on Nicheng only, scored on Nation and Shanghai (all their photos, training and validation)

In [ ]:
src = pd.concat([pd.read_csv(f) for f in glob.glob(f'{DD}/regular_fundus_images/*/regular-fundus-source-*.csv')])
site = dd.image_id.map(src.set_index('image_id').Source).fillna('?').values
home, other, home_val = tr & (site == 'Nicheng'), site != 'Nicheng', va & (site == 'Nicheng')
c2 = cc.Coach.fit(X[home], chk[home], {k: v[home] for k, v in Y.items()})
P2 = c2.probs(X, chk)
rows = []
for j, h in enumerate(cc.Coach.HEADS):
    a1 = auc_ci(Y[h][home_val], P2[home_val, j]); a2 = auc_ci(Y[h][other], P2[other, j])
    rows.append({'check': h, 'Nicheng validation': f'{a1[0]:.3f} ({a1[1]:.2f}-{a1[2]:.2f})',
                 f'Nation + Shanghai (n={other.sum()})': f'{a2[0]:.3f} ({a2[1]:.2f}-{a2[2]:.2f})',
                 'photos with it there': int(Y[h][other].sum())})
say('\n== 2. Other DeepDRiD hospitals: AUROC (95% CI) ==')
say(pd.DataFrame(rows).to_string(index=False))

## 3. New camera: IDRiD
IDRiD photos were picked for grading, so nearly all should be kept. The damaged copies show whether the right problem is named.
Exposure has no head of its own, so for dark or washed-out photos only the retake rate counts.

In [ ]:
Vi = coach.verdict(G['idr_X'], G['idr_C'])
say(f"\n== 3. New camera (IDRiD) ==\nRetake asked for {np.mean([v['retake'] for v in Vi]):.0%} of {len(Vi)} IDRiD photos picked for grading "
    f"(DeepDRiD test good photos: {(flag & ~bad).sum() / (~bad).sum():.0%})")
Vc = coach.verdict(G['cor_X'], G['cor_C'])
rows = []
for g in GROUPS:
    idx = np.where(G['cor_group'] == g)[0]
    r = [Vc[i] for i in idx if Vc[i]['retake']]
    rows.append({'damage': g, 'retake asked': f"{len(r)}/{len(idx)}",
                 'top problem named (of retakes)': ', '.join(f'{k} {sum(v["problems"][0] == k for v in r)}' for k in cc.PROBLEMS)})
say(pd.DataFrame(rows).to_string(index=False))

## 4. Guard: does the photo look like a regular fundus photo at all?
Energy = Mahalanobis distance to ordinary fundus photos, a Gaussian energy model fitted on positives only, with no negatives
(LeCun's "regularized" family). Two versions:
- **one camera:** fitted on DeepDRiD training photos only.
- **two cameras:** add 200 unlabeled IDRiD photos. This is how a clinic would onboard its own camera: no labels, just ordinary photos.

Each threshold is the 99th percentile on held-out fundus photos, so about 1% of real fundus photos get turned away.

In [ ]:
ev = list(G['ev_names'])
fundus_web = np.array([n == 'retina' for n in ev]); everyday = ~fundus_web
idr_ref, idr_cal, idr_test = np.arange(200), np.arange(200, 300), np.arange(300, len(G['idr_X']))
sets = {'DeepDRiD test (same camera)': (G['te_X'], True), 'IDRiD photos 300+ (new camera)': (G['idr_X'][idr_test], True),
        'IDRiD damaged copies': (G['cor_X'], True), 'fundus photo from the web (skimage)': (G['ev_X'][fundus_web], True),
        'widefield photos (DeepDRiD UWF)': (G['uwf_X'], False), 'everyday pictures + black frame': (G['ev_X'][everyday], False)}
guards = {'one camera': (X[tr], X[va]),
          'two cameras': (np.r_[X[tr], G['idr_X'][idr_ref]], np.r_[X[va], G['idr_X'][idr_cal]])}
rows, gfit = [], {}
for gname, (ref, cal) in guards.items():
    g = cc.Coach(coach.mu, coach.sd, coach.W, coach.b, coach.thr).set_guard(ref, cal)
    gfit[gname] = g
    for sname, (Xs, is_fundus) in sets.items():
        if len(Xs) == 0: continue
        rows.append({'guard': gname, 'photos': f'{sname} (n={len(Xs)})', 'should be': 'accepted' if is_fundus else 'turned away',
                     'accepted': f'{(g.energy(Xs) <= g.g_thr).mean():.0%}'})
say('\n== 4. Energy guard ==')
say(pd.DataFrame(rows).pivot(index=['photos', 'should be'], columns='guard', values='accepted').to_string())
g = gfit['two cameras']
for n, e in zip(ev, g.energy(G['ev_X'])):
    say(f'  {n:22s} energy {e:8.2f}  (threshold {g.g_thr:.2f})')

## 5. Save the tool's model, and look at a few verdicts

In [ ]:
import matplotlib.pyplot as plt
coach = gfit['two cameras']
coach.save(f'{D}/quality/coach_model.npz')
say(f"\nSaved quality/coach_model.npz: retake threshold {coach.thr[0]:.2f}, problem thresholds "
    + ', '.join(f'{k} {t:.2f}' for k, t in zip(cc.PROBLEMS, coach.thr[1:])) + f', guard energy <= {coach.g_thr:.2f}')

Vt = coach.verdict(G['te_X'], G['te_C'])
pick = []
for want in ['keep', 'focus', 'aim', 'obstruction']:
    for i, v in enumerate(Vt):
        if (want == 'keep' and v['looks_like_fundus'] and not v['retake'] and not bad[i]) or (v['retake'] and v['problems'][:1] == [want] and want in truth[i]):
            pick.append(('test', i)); break
pick += [('uwf', 0), ('ev', ev.index('chelsea') if 'chelsea' in ev else 0)]
import skimage.data as sd
def picture(kind, i):
    if kind == 'test': return Image.open(te_paths[i]), G['te_X'], G['te_C'], f"DeepDRiD test {te.image_id[i]}, truth: " + (
        ('not good enough (' + (', '.join(sorted(truth[i])) or 'no single problem') + ')') if bad[i] else 'good')
    if kind == 'uwf': return Image.open(uwf_paths[i]), G['uwf_X'], G['uwf_C'], 'widefield photo, truth: not a regular fundus photo'
    return Image.fromarray(np.asarray(getattr(sd, ev[i])())), G['ev_X'], G['ev_C'], f'everyday picture ({ev[i]})'

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (kind, i) in zip(axes.ravel(), pick):
    img, Xi, Ci, label = picture(kind, i)
    v = coach.verdict(Xi[i:i + 1], Ci[i:i + 1])[0]
    img.thumbnail((600, 600)); ax.imshow(img); ax.axis('off')
    if not v['looks_like_fundus']: head, text = 'NOT A FUNDUS PHOTO', cc.NOT_FUNDUS
    elif v['retake']: head, text = 'RETAKE: ' + ', '.join(v['problems']), cc.template(v['problems'])
    else: head, text = 'KEEP', 'Good enough to grade.'
    ax.set_title(f"{head}\n{text}\n{label}\n(p not good {v['p_not_good']:.2f}, energy {v['energy']:.1f})", fontsize=9)
for ax in axes.ravel()[len(pick):]: ax.axis('off')
plt.tight_layout(); plt.savefig(f'{D}/quality/coach_examples.png', dpi=110); plt.show()

## Optional (paid): Muse Spark words the instruction for the problems the heads found
Off by default. With `RUN_MUSE = True` it makes at most **8 requests** and stops before **$0.08**, using
`reasoning_effort='minimal'` and 768 px photos. It needs `MODEL_API_KEY` in Colab Secrets.

In [ ]:
RUN_MUSE = False
if RUN_MUSE:
    sh('pip install -q openai')
    from google.colab import userdata
    from openai import OpenAI
    words = cc.Wording(OpenAI(base_url='https://api.meta.ai/v1', api_key=userdata.get('MODEL_API_KEY'), timeout=60),
                       max_requests=8, max_usd=0.08)
    chosen = []
    for k in cc.PROBLEMS:
        chosen += [i for i, v in enumerate(Vt) if v['retake'] and v['problems'] == [k] and k in truth[i]][:2]
    chosen += [i for i, v in enumerate(Vt) if v['retake'] and len(v['problems']) == 2][:2]
    out = []
    for i in chosen:
        r = words(te_paths[i], Vt[i]['problems'])
        out.append(dict(image_id=te.image_id[i], problems=Vt[i]['problems'], instruction=r['text'], source=r['source'], cost=r['cost']))
        print(f"{te.image_id[i]:8s} {str(Vt[i]['problems']):28s} [{r['source']}] {r['text']}")
    pd.DataFrame(out).to_csv(f'{D}/quality/muse_wording_pilot.csv', index=False)
    say(f'\nMuse Spark wording: {words.n} requests, ${words.spent:.4f} total')